# TrackMate R=2.5 slice 40/80 viewer

`outputs/TMoptimization/r2p5_median_off_slices_040_080` の80画像を確認する専用viewerです。

- 10動物 × L/R × ventral/dorsal × Slice 40/80
- LoG radius 2.5、Median OFF、Q ≥ 150
- Contrast/SNRの負値を0にして全特徴量をlog1p変換後、MAYxxL/R block内でd/v・Slice 40/80をまとめてRobust Z化
- 共通Robust Z上下限の積集合から、再現可能なランダム標本を表示
- Dataset・Slice・前後ボタンで全80画像を移動
- Preprocessed / Raw背景、Zoom、XYスライダー、パン操作に対応

初期状態ではRobust Zフィルタを適用せず、Q ≥ 150からランダムに10,000点を描画します。

In [1]:
from functools import lru_cache
from pathlib import Path
import re
import xml.etree.ElementTree as ET
import zlib

import ipywidgets as widgets
from IPython.display import clear_output, display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile as tiff

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise RuntimeError('Project root containing src/ was not found.')

RESULT_ROOT = (
    PROJECT_ROOT / 'outputs' / 'TMoptimization' /
    'r2p5_median_off_slices_040_080'
)
SUMMARY_PATH = RESULT_ROOT / 'robust_z_log1p_floor0_summary.csv'
DETECTION_SUMMARY_PATH = RESULT_ROOT / 'detection_summary.csv'
for required_path in [SUMMARY_PATH, DETECTION_SUMMARY_PATH]:
    if not required_path.is_file():
        raise FileNotFoundError(required_path)

SUMMARY = pd.read_csv(str(SUMMARY_PATH))
DETECTION_SUMMARY = pd.read_csv(str(DETECTION_SUMMARY_PATH))
if len(SUMMARY) != 80:
    raise RuntimeError('Expected 80 Robust Z rows, found {}'.format(len(SUMMARY)))
if len(DETECTION_SUMMARY) != 80:
    raise RuntimeError('Expected 80 detection rows, found {}'.format(len(DETECTION_SUMMARY)))

ROBUST_FEATURES = [
    ('Contrast', 'ROBUST_Z_CONTRAST_CH1'),
    ('SNR', 'ROBUST_Z_SNR_CH1'),
    ('SD', 'ROBUST_Z_STD_INTENSITY_CH1'),
    ('CV', 'ROBUST_Z_CV'),
    ('Mean', 'ROBUST_Z_MEAN_INTENSITY_CH1'),
    ('Min', 'ROBUST_Z_MIN_INTENSITY_CH1'),
    ('Quality', 'ROBUST_Z_QUALITY'),
    ('Max', 'ROBUST_Z_MAX_INTENSITY_CH1'),
    ('Median', 'ROBUST_Z_MEDIAN_INTENSITY_CH1'),
    ('Total', 'ROBUST_Z_TOTAL_INTENSITY_CH1'),
]

DATASETS = sorted(SUMMARY['dataset'].unique())
SLICES = sorted(int(value) for value in SUMMARY['slice'].unique())
if len(DATASETS) != 40 or SLICES != [40, 80]:
    raise RuntimeError('Unexpected dataset or slice grid.')

ROW_INDEX = {}
for _, row in SUMMARY.iterrows():
    key = (row['dataset'], int(row['slice']))
    if key in ROW_INDEX:
        raise RuntimeError('Duplicate result row: {}'.format(key))
    ROW_INDEX[key] = row
DETECTION_INDEX = {
    (row['dataset'], int(row['slice'])): row
    for _, row in DETECTION_SUMMARY.iterrows()
}

IMAGE_KEYS = [(dataset, slice_number) for dataset in DATASETS for slice_number in SLICES]
if len(IMAGE_KEYS) != 80 or set(IMAGE_KEYS) != set(ROW_INDEX):
    raise RuntimeError('The 40 × 2 image grid is incomplete.')

def current_path(path_string):
    path = Path(path_string)
    parts = list(path.parts)
    if 'outputs' in parts:
        index = parts.index('outputs')
        return PROJECT_ROOT.joinpath(*parts[index:])
    return path

def row_for(dataset, slice_number):
    return ROW_INDEX[(dataset, int(slice_number))]

def processed_path(dataset, slice_number):
    row = DETECTION_INDEX[(dataset, int(slice_number))]
    return current_path(row['input_tif'])

def raw_path(dataset, slice_number):
    processed = processed_path(dataset, slice_number)
    stem = re.sub(r'_\d{3}$', '', processed.stem)
    return PROJECT_ROOT / 'data' / (stem + '.tif')

def xml_path(dataset, slice_number):
    return current_path(row_for(dataset, slice_number)['output_xml'])

for dataset, slice_number in IMAGE_KEYS:
    if not processed_path(dataset, slice_number).is_file():
        raise FileNotFoundError(processed_path(dataset, slice_number))
    if not raw_path(dataset, slice_number).is_file():
        raise FileNotFoundError(raw_path(dataset, slice_number))
    if not xml_path(dataset, slice_number).is_file():
        raise FileNotFoundError(xml_path(dataset, slice_number))

print('Project:', PROJECT_ROOT)
print('Datasets:', len(DATASETS))
print('Images:', len(IMAGE_KEYS))

Project: C:\workspace\LSFM_pp
Datasets: 40
Images: 80


In [2]:
@lru_cache(maxsize=8)
def read_spots(xml_path_string, expected_count):
    count = int(expected_count)
    spots = np.empty((count, 3 + len(ROBUST_FEATURES)), dtype=np.float32)
    index = 0
    for _, element in ET.iterparse(xml_path_string, events=('end',)):
        if element.tag == 'Spot':
            if index >= count:
                raise RuntimeError('More spots than expected in {}'.format(xml_path_string))
            spots[index, 0] = float(element.attrib['POSITION_X'])
            spots[index, 1] = float(element.attrib['POSITION_Y'])
            spots[index, 2] = float(element.attrib['QUALITY'])
            for feature_index, (_, attribute) in enumerate(ROBUST_FEATURES):
                spots[index, 3 + feature_index] = float(element.attrib[attribute])
            index += 1
        element.clear()
    if index != count:
        raise RuntimeError(
            'Spot count mismatch in {}: expected {}, read {}'.format(
                xml_path_string, count, index
            )
        )
    return spots

@lru_cache(maxsize=4)
def read_processed_image(path_string):
    return tiff.imread(path_string)

@lru_cache(maxsize=4)
def read_raw_image(path_string, page_index):
    with tiff.TiffFile(path_string) as tif_file:
        return tif_file.pages[int(page_index)].asarray()

def read_background(dataset, slice_number, source):
    if source == 'processed':
        return read_processed_image(str(processed_path(dataset, slice_number).resolve()))
    return read_raw_image(
        str(raw_path(dataset, slice_number).resolve()), int(slice_number) - 1
    )

def random_sample_spots(spots, active_filters, sample_n, seed):
    mask = np.ones(len(spots), dtype=bool)
    for feature_index, lower, upper, _ in active_filters:
        values = spots[:, 3 + feature_index]
        mask &= (values > float(lower)) & (values < float(upper))
    eligible = np.flatnonzero(mask)
    eligible_count = len(eligible)
    if eligible_count > int(sample_n):
        rng = np.random.RandomState(int(seed) & 0xffffffff)
        eligible = rng.choice(eligible, size=int(sample_n), replace=False)
    return spots[eligible], eligible_count

def clipped_view_bounds(center, full_size, zoom):
    view_size = float(full_size) / float(zoom)
    lower = float(center) - view_size / 2.0
    lower = min(max(lower, 0.0), max(float(full_size) - view_size, 0.0))
    return lower, lower + view_size

In [ ]:
dataset_widget = widgets.Dropdown(
    options=DATASETS, value=DATASETS[0], description='Dataset:',
    layout=widgets.Layout(width='55%')
)
slice_widget = widgets.ToggleButtons(
    options=[('Slice 40', 40), ('Slice 80', 80)], value=40, description='Slice:'
)
previous_button = widgets.Button(description='← Previous', icon='step-backward')
next_button = widgets.Button(description='Next →', icon='step-forward')
image_index_widget = widgets.HTML()
background_widget = widgets.ToggleButtons(
    options=[('Preprocessed', 'processed'), ('Raw', 'raw')],
    value='processed', description='Background:'
)
quality_note_widget = widgets.HTML('<b>Q ≥ 150 fixed</b>')
sample_n_widget = widgets.BoundedIntText(
    value=10000, min=100, max=100000, step=100, description='Random N:'
)
robust_filter_widgets = []
robust_filter_rows = []
for feature_index, (label, _) in enumerate(ROBUST_FEATURES):
    enabled = widgets.Checkbox(
        value=False, description=label,
        layout=widgets.Layout(width='115px'), indent=False
    )
    bounds = widgets.FloatRangeSlider(
        value=(-3.0, 3.0), min=-10.0, max=10.0, step=0.1,
        description='Z:', disabled=True, continuous_update=False,
        readout_format='.1f', layout=widgets.Layout(width='75%')
    )
    robust_filter_widgets.append((feature_index, label, enabled, bounds))
    robust_filter_rows.append(widgets.HBox([enabled, bounds]))
robust_filters_accordion = widgets.Accordion(
    children=[widgets.VBox(robust_filter_rows)], selected_index=None
)
robust_filters_accordion.set_title(0, 'Common Robust Z filters (all disabled)')
marker_size_widget = widgets.FloatSlider(
    value=5.0, min=1.0, max=30.0, step=1.0, description='Marker:',
    continuous_update=False, layout=widgets.Layout(width='45%')
)
marker_alpha_widget = widgets.FloatSlider(
    value=0.7, min=0.1, max=1.0, step=0.1, description='Alpha:',
    continuous_update=False, layout=widgets.Layout(width='45%')
)
marker_color_widget = widgets.Dropdown(
    options=['lime', 'red', 'cyan', 'yellow', 'magenta'],
    value='lime', description='Color:'
)
contrast_widget = widgets.FloatRangeSlider(
    value=(0.5, 99.7), min=0.0, max=100.0, step=0.1,
    description='Percentile:', continuous_update=False,
    readout_format='.1f', layout=widgets.Layout(width='75%')
)
zoom_widget = widgets.SelectionSlider(
    options=[1, 1.5, 2, 3, 4, 6, 8, 12, 16], value=1,
    description='Zoom:', continuous_update=False,
    layout=widgets.Layout(width='45%')
)
center_x_widget = widgets.IntSlider(
    value=2048, min=0, max=4095, step=1, description='Center X:',
    continuous_update=False, layout=widgets.Layout(width='80%')
)
center_y_widget = widgets.IntSlider(
    value=1080, min=0, max=2159, step=1, description='Center Y:',
    continuous_update=False, layout=widgets.Layout(width='80%')
)
figure_width_widget = widgets.FloatSlider(
    value=11.0, min=5.0, max=16.0, step=0.5, description='Figure:',
    continuous_update=False, layout=widgets.Layout(width='45%')
)
reset_view_button = widgets.Button(description='Reset view', icon='refresh')
refresh_button = widgets.Button(description='Refresh', icon='play')
pan_left_button = widgets.Button(description='←', layout=widgets.Layout(width='45px'))
pan_right_button = widgets.Button(description='→', layout=widgets.Layout(width='45px'))
pan_up_button = widgets.Button(description='↑', layout=widgets.Layout(width='45px'))
pan_down_button = widgets.Button(description='↓', layout=widgets.Layout(width='45px'))
status_widget = widgets.HTML()
viewer_output = widgets.Output()
_updating = False

def current_key():
    return (dataset_widget.value, int(slice_widget.value))

def current_index():
    return IMAGE_KEYS.index(current_key())

def active_robust_filters():
    active = []
    for feature_index, label, enabled, bounds in robust_filter_widgets:
        if enabled.value:
            lower, upper = bounds.value
            active.append((feature_index, float(lower), float(upper), label))
    return active

def update_index_label():
    image_index_widget.value = '<b>Image {:02d} / 80</b>'.format(current_index() + 1)

def reset_view(_=None, render=True):
    global _updating
    dataset, slice_number = current_key()
    image = read_background(dataset, slice_number, background_widget.value)
    height, width = image.shape
    _updating = True
    center_x_widget.max = width - 1
    center_y_widget.max = height - 1
    center_x_widget.value = width // 2
    center_y_widget.value = height // 2
    zoom_widget.value = 1
    _updating = False
    update_index_label()
    if render:
        render_view()

def navigate(delta):
    global _updating
    new_index = (current_index() + int(delta)) % len(IMAGE_KEYS)
    dataset, slice_number = IMAGE_KEYS[new_index]
    _updating = True
    dataset_widget.value = dataset
    slice_widget.value = slice_number
    _updating = False
    reset_view(render=False)
    render_view()

def pan_view(dx, dy):
    global _updating
    dataset, slice_number = current_key()
    image = read_background(dataset, slice_number, background_widget.value)
    height, width = image.shape
    zoom = float(zoom_widget.value)
    step_x = max(1, int((width / zoom) * 0.25))
    step_y = max(1, int((height / zoom) * 0.25))
    _updating = True
    center_x_widget.value = int(np.clip(center_x_widget.value + dx * step_x, 0, width - 1))
    center_y_widget.value = int(np.clip(center_y_widget.value + dy * step_y, 0, height - 1))
    _updating = False
    render_view()

def render_view(_=None):
    if _updating:
        return
    dataset, slice_number = current_key()
    try:
        image = read_background(dataset, slice_number, background_widget.value)
        summary_row = row_for(dataset, slice_number)
        spots = read_spots(
            str(xml_path(dataset, slice_number).resolve()),
            int(summary_row['spots_q150'])
        )
        active_filters = active_robust_filters()
        filter_signature = tuple(
            (feature_index, lower, upper)
            for feature_index, lower, upper, _ in active_filters
        )
        seed_text = '{}|{}|{}'.format(dataset, slice_number, filter_signature)
        seed = zlib.crc32(seed_text.encode('utf-8')) & 0xffffffff
        shown, eligible_count = random_sample_spots(
            spots, active_filters, sample_n_widget.value, seed
        )
        low_p, high_p = contrast_widget.value
        vmin, vmax = np.percentile(image, [low_p, high_p])
        if vmax <= vmin:
            vmax = vmin + 1
        height, width = image.shape
        zoom = float(zoom_widget.value)
        x0, x1 = clipped_view_bounds(center_x_widget.value, width, zoom)
        y0, y1 = clipped_view_bounds(center_y_widget.value, height, zoom)
        figure_width = float(figure_width_widget.value)
        figure_height = min(11.0, max(3.0, figure_width * (y1 - y0) / max(x1 - x0, 1)))
        fig, ax = plt.subplots(figsize=(figure_width, figure_height), dpi=120)
        ax.imshow(image, cmap='gray', vmin=vmin, vmax=vmax, origin='upper')
        ax.scatter(
            shown[:, 0], shown[:, 1], s=marker_size_widget.value,
            facecolors='none', edgecolors=marker_color_widget.value,
            linewidths=0.7, alpha=marker_alpha_widget.value,
        )
        ax.set_xlim(x0, x1)
        ax.set_ylim(y1, y0)
        ax.set_title(
            '{} | Slice {:03d} | Q≥150 | R=2.5 Median OFF | {} | zoom {}x'.format(
                dataset, slice_number, background_widget.label, zoom_widget.value
            )
        )
        ax.set_xlabel('X (pixel)')
        ax.set_ylabel('Y (pixel)')
        fig.tight_layout()
        update_index_label()
        if active_filters:
            filter_text = '; '.join(
                '{} ({:.1f}, {:.1f})'.format(label, lower, upper)
                for _, lower, upper, label in active_filters
            )
        else:
            filter_text = 'none'
        status_widget.value = (
            '<b>{}</b> Slice {:03d} | Q≥150 spots {:,} | Robust Z filters: {} '
            '| eligible {:,} | random sample {:,}'.format(
                dataset, slice_number, len(spots), filter_text,
                eligible_count, len(shown)
            )
        )
        with viewer_output:
            clear_output(wait=True)
            display(fig)
            plt.close(fig)
    except Exception as error:
        status_widget.value = '<b style="color:red">{}: {}</b>'.format(type(error).__name__, error)
        with viewer_output:
            clear_output(wait=True)
            print(type(error).__name__ + ':', error)

def change_image(_=None):
    if _updating:
        return
    reset_view(render=False)
    render_view()

def change_filter_enabled(change, bounds):
    bounds.disabled = not bool(change['new'])
    render_view()

previous_button.on_click(lambda _: navigate(-1))
next_button.on_click(lambda _: navigate(1))
reset_view_button.on_click(reset_view)
refresh_button.on_click(render_view)
pan_left_button.on_click(lambda _: pan_view(-1, 0))
pan_right_button.on_click(lambda _: pan_view(1, 0))
pan_up_button.on_click(lambda _: pan_view(0, -1))
pan_down_button.on_click(lambda _: pan_view(0, 1))
dataset_widget.observe(change_image, names='value')
slice_widget.observe(change_image, names='value')
background_widget.observe(change_image, names='value')
for widget in [
    sample_n_widget, marker_size_widget,
    marker_alpha_widget, marker_color_widget, contrast_widget, zoom_widget,
    center_x_widget, center_y_widget, figure_width_widget,
]:
    widget.observe(render_view, names='value')
for _, _, enabled, bounds in robust_filter_widgets:
    enabled.observe(
        lambda change, bounds=bounds: change_filter_enabled(change, bounds),
        names='value'
    )
    bounds.observe(render_view, names='value')

controls = widgets.VBox([
    widgets.HBox([dataset_widget, slice_widget, image_index_widget]),
    widgets.HBox([previous_button, next_button, background_widget]),
    widgets.HBox([quality_note_widget, sample_n_widget, refresh_button]),
    robust_filters_accordion,
    widgets.HBox([marker_size_widget, marker_alpha_widget, marker_color_widget]),
    contrast_widget,
    widgets.HBox([zoom_widget, figure_width_widget, reset_view_button]),
    center_x_widget,
    center_y_widget,
    widgets.HBox([pan_left_button, pan_right_button, pan_up_button, pan_down_button]),
    status_widget,
])

reset_view(render=False)
display(controls, viewer_output)
render_view()

Output()